# energy-audit · Step 3 · Magenta RT 2 (mrt2_base) smoke test

Six clips: all six seeds of one prompt, plus the first seed run a second time.

**Chosen before the run** with `numpy.random.default_rng(20261108)`: the prompt `rock_L1_verb`
("instrumental rock track, very calm"); the 3 listening clips are seeds s0, s1, s3.

Checks:
1. JAX (CUDA 12 build) sees the GPU.
2. Every clip: 48 kHz, 2 channels, exactly 30.000 s, PCM_16, not silent (asserted inside `gen/mrt2.py`).
3. The six seeds give six different clips (distinct PCM hashes).
4. The same seed run twice gives identical audio.
5. Time per clip, and the projection for 300 clips (stop rule: over ~10 GPU-hours).

Output tab: `listen/` (3 WAVs + 6 spectrograms), `smoke_report.json`, `manifest_mrt2.csv`, logs, `pip_freeze.txt`.
The harness is fetched from the public repo at commit `c2b94de` and hash-checked.

In [ ]:
# 1. Environment + pinned install + JAX GPU check (in a subprocess, so this kernel never holds GPU memory)
import subprocess, sys, os, json, time, socket, platform
T0 = time.time()
W = "/kaggle/working"; os.makedirs(W, exist_ok=True)
socket.getaddrinfo("huggingface.co", 443); print("internet: OK")
PINS = ["magenta-rt==2.0.3", "jax[cuda12]==0.11.0", "flax==0.12.8"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS, "pyloudnorm"], check=True)
open(f"{W}/pip_freeze.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                         capture_output=True, text=True).stdout)
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
probe = subprocess.run([sys.executable, "-c",
    "import jax,jaxlib,flax,magenta_rt,json;d=jax.devices();"
    "x=jax.numpy.ones((256,256));y=float((x@x).sum());"
    "print(json.dumps(dict(jax=jax.__version__,jaxlib=jaxlib.__version__,flax=flax.__version__,"
    "magenta_rt=magenta_rt.__version__,devices=[str(i) for i in d],platform=d[0].platform,matmul_ok=y==256**3)))"],
    capture_output=True, text=True)
print(probe.stdout[-2000:], probe.stderr[-3000:])
ENV = dict(python=platform.python_version(), gpu=smi, install_pins=PINS,
           kaggle_docker_image=os.environ.get("KAGGLE_DOCKER_IMAGE", "not exposed (pinned in kernel metadata)"))
try: ENV.update(json.loads(probe.stdout.strip().splitlines()[-1]))
except Exception: ENV["probe_error"] = probe.stderr[-3000:]
JAX_GPU_OK = ENV.get("platform") == "gpu" and ENV.get("matmul_ok") is True
ENV["jax_cuda12_on_gpu"] = JAX_GPU_OK
json.dump(ENV, open(f"{W}/env.json", "w"), indent=1); print(json.dumps(ENV, indent=1))


In [ ]:
# 2. Fetch the harness + frozen matrix at the pinned commit; verify hashes
import hashlib, urllib.request
COMMIT = "c2b94de58bf6efb27e5de58d37e44d89b8e806c6"
WANT = {"gen/mrt2.py": "dc58477893d9c265d30616f179c60e8bee827ac10c0285aca1ad2d54f71138b1",
        "data/prompts.csv": "9bf219753c62e83ecc47181002d5a49d6e83ff15ee7e60a64f572be12f0e2a2e"}
R = "/tmp/repo"
for rel, h in WANT.items():
    os.makedirs(os.path.dirname(f"{R}/{rel}"), exist_ok=True)
    b = urllib.request.urlopen(f"https://raw.githubusercontent.com/ConnerFarrell/energy-audit/{COMMIT}/{rel}").read()
    assert hashlib.sha256(b).hexdigest() == h, rel
    open(f"{R}/{rel}", "wb").write(b)
print("harness + prompts: hash OK at", COMMIT[:7])


In [ ]:
# 3. Generate: 6 seeds of rock_L1_verb, then seed s0 again (--repeat-first)
SMOKE_PROMPT, LISTEN = "rock_L1_verb", [0, 1, 3]
OUTD = "/tmp/smoke"
RC = None
if JAX_GPU_OK:
    with open(f"{W}/smoke_log.txt", "w") as log:
        p = subprocess.Popen([sys.executable, f"{R}/gen/mrt2.py", "--prompts", f"{R}/data/prompts.csv",
                              "--out", OUTD, "--magenta-home", "/tmp/magenta", "--size", "mrt2_base",
                              "--only", SMOKE_PROMPT, "--seeds", "0,1,2,3,4,5", "--repeat-first"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); log.flush()
            if line.startswith("[mrt2]") or "Error" in line or "Traceback" in line: print(line, end="")
        RC = p.wait()
    print("harness exit code:", RC)
else:
    print("SKIPPED: JAX CUDA 12 build does not see the GPU (stop condition 1)")


In [ ]:
# 4. Checks + timing projection -> smoke_report.json
import pandas as pd, numpy as np, re, shutil
rep = dict(jax_cuda12_on_gpu=JAX_GPU_OK, harness_exit_code=RC, smoke_prompt=SMOKE_PROMPT,
           listen_seeds=LISTEN, listen_choice_rng="numpy.random.default_rng(20261108), drawn before the run")
man = f"{OUTD}/manifest_mrt2.csv"
if os.path.exists(man):
    M = pd.read_csv(man); shutil.copy(man, f"{W}/manifest_mrt2.csv")
    log = open(f"{W}/smoke_log.txt").read()
    m_rep = re.search(r"REPEAT (\S+): (IDENTICAL|DIFFERENT)", log)
    m_dl = re.search(r"download (\d+)s, load\+compile (\d+)s", log)
    rep.update(
        clips=len(M),
        audit_all_pass=bool(len(M) == 6 and (M.sample_rate == 48000).all() and (M.channels == 2).all()
                            and (M.n_samples == 1_440_000).all()),
        distinct_seeds=int(M.seed.nunique()), distinct_pcm=int(M.pcm_sha256.nunique()),
        six_seeds_six_different_clips=bool(M.pcm_sha256.nunique() == 6 == len(M)),
        same_seed_twice_identical=(m_rep.group(2) == "IDENTICAL") if m_rep else None,
        repeat_clip=m_rep.group(1) if m_rep else None,
        gen_s_per_clip=M.gen_s.round(1).tolist(),
        gen_s_mean_all=round(float(M.gen_s.mean()), 1),
        gen_s_mean_excl_first=round(float(M.gen_s.iloc[1:].mean()), 1),
        download_s=int(m_dl.group(1)) if m_dl else None, load_compile_s=int(m_dl.group(2)) if m_dl else None,
        peak=M.peak.round(3).tolist(), rms_dbfs=M.rms_dbfs.round(1).tolist())
    per = rep["gen_s_mean_excl_first"]
    over = (rep["download_s"] or 0) + (rep["load_compile_s"] or 0)
    rep["projected_300_clip_gpu_h"] = round((300 * per + over) / 3600, 2)
    rep["within_10_gpu_h"] = rep["projected_300_clip_gpu_h"] <= 10.0
rep["notebook_wall_min"] = round((time.time() - T0) / 60, 1)
json.dump(rep, open(f"{W}/smoke_report.json", "w"), indent=1); print(json.dumps(rep, indent=1))


In [ ]:
# 5. Listening pack: the 3 pre-chosen WAVs; one spectrogram per clip (all 6). Panels as in Step 2:
# (1) as generated (mono sum, native rate); (2) scorer input (librosa mono, 22.05 kHz, first 30 s);
# (3) scorer input after -14 LUFS normalization.
import soundfile as sf, librosa, librosa.display, pyloudnorm as pyln
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
L = f"{W}/listen"; os.makedirs(L, exist_ok=True)
def spec(ax, y, sr, title):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=2048, hop_length=512)), ref=np.max)
    img = librosa.display.specshow(S, sr=sr, hop_length=512, x_axis="time", y_axis="log", ax=ax, cmap="magma", vmin=-80, vmax=0)
    ax.set_title(title, fontsize=9); return img
for k in range(6):
    b = f"mrt2__{SMOKE_PROMPT}__s{k}.wav"; p = f"{OUTD}/audio/{b}"
    if not os.path.exists(p): continue
    info = sf.info(p)
    x, sr0 = sf.read(p, dtype="float32", always_2d=True); y0 = x.mean(axis=1)
    y, sr = librosa.load(p, sr=22050, mono=True, duration=30)
    lufs = pyln.Meter(sr).integrated_loudness(y)
    yn = np.clip(y * 10 ** ((-14.0 - lufs) / 20.0), -1, 1)
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    spec(axes[0], y0, sr0, f"as generated: {info.samplerate} Hz x {info.channels} ch, {info.duration:.2f} s (mono sum shown)")
    spec(axes[1], y, sr, f"scorer input: mono, 22.05 kHz, first 30 s  (integrated loudness {lufs:.1f} LUFS)")
    img = spec(axes[2], yn, sr, "scorer input after -14 LUFS normalization")
    fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6)
    fig.suptitle(b.replace(".wav", ""), y=0.995)
    fig.savefig(f"{L}/{b.replace('.wav', '')}_spectrogram.png", dpi=110, bbox_inches="tight"); plt.close(fig)
    if k in LISTEN: shutil.copy(p, f"{L}/{b}")
print(sorted(os.listdir(L))); print(sorted(os.listdir(W)))
